# Rydberg blockade and antiferromagnetic order

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/official-dvl/zksf/blob/main/examples/tutorials/rydberg-antiferromagnetic-chain.ipynb)

A chain of neutral atoms, driven slowly enough, settles into an alternating pattern:
excited, ground, excited, ground. This notebook measures **how much blockade you need**
for that to happen, and shows what you get when there is not enough.

Unlike the gate-circuit notebooks here, **everything runs locally and free**. Pulser's
simulator is an ordinary pip package, so no account or token is needed until the
optional last section.


## The one number that matters

An atom in a Rydberg state shifts its neighbours' energy enough that the same laser can
no longer excite them. The distance over which that holds is the **blockade radius**, `Rb`.
It depends on how hard you drive the atoms: a stronger drive is harder to suppress.

What decides the physics is `Rb / a`, the ratio of that radius to the atom spacing `a`.

- `Rb/a > 1`: each atom blocks its neighbour, so the most atoms you can excite is every
  other one. That is the alternating pattern.
- `Rb/a < 1`: neighbours stop blocking each other, and nothing enforces the alternation.


In [ ]:
!pip install -q pulser


In [ ]:
import numpy as np
import pulser
from pulser import Pulse, Register, Sequence
from pulser.devices import AnalogDevice
from pulser.backend import BitStrings
from pulser_simulation import QutipBackendV2, QutipConfig

N_ATOMS = 7
OMEGA = 4 * np.pi                 # drive strength, rad/us
SHOTS = 1000

Rb = AnalogDevice.rydberg_blockade_radius(OMEGA)
print(f'blockade radius at this drive: {Rb:.2f} um')


## Building the sequence

A global Rydberg beam drives all seven atoms at once. The sweep has three phases:
amplitude ramps up, detuning sweeps from far negative to positive while the amplitude
holds, then amplitude ramps down. It has to be slow enough that the system follows the
instantaneous ground state, which is what carries it into the ordered arrangement.

`mimic_qpu=True` applies the same validation the real hardware applies, so a sequence
this notebook accepts is one the QPU would accept too.


In [ ]:
def afm_chain(spacing_um, n=N_ATOMS, omega=OMEGA):
    """A 1D chain driven through an adiabatic detuning sweep."""
    coords = np.array([[i * spacing_um, 0.0] for i in range(n)])
    coords -= coords.mean(axis=0)        # centre it; the device limits the radius
    reg = Register.from_coordinates(coords, prefix='q').with_automatic_layout(AnalogDevice)

    seq = Sequence(reg, AnalogDevice)
    seq.declare_channel('ising', 'rydberg_global')
    d0, df = -3 * omega, omega
    seq.add(Pulse.ConstantDetuning(
        pulser.waveforms.RampWaveform(400, 0.0, omega), d0, 0.0), 'ising')
    seq.add(Pulse.ConstantAmplitude(
        omega, pulser.waveforms.RampWaveform(3200, d0, df), 0.0), 'ising')
    seq.add(Pulse.ConstantDetuning(
        pulser.waveforms.RampWaveform(400, omega, 0.0), df, 0.0), 'ising')
    return seq


def run_local(seq, shots=SHOTS):
    """Exact simulation on this machine. No account, no queue, no cost."""
    cfg = QutipConfig(observables=[BitStrings(num_shots=shots)])
    res = QutipBackendV2(seq, config=cfg, mimic_qpu=True).run()
    return {str(k): int(v) for k, v in res.final_bitstrings.items()}


seq = afm_chain(5.2)
print(f'{len(seq.register.qubits)} atoms, {seq.get_duration()} ns')
seq.draw('input')


## One run

At 5.2 um the blockade radius is 6.40 um, so `Rb/a` is 1.23: each atom reaches its
neighbour and stops short of the one beyond. A bit reads 1 when that atom finished in the
Rydberg state, so perfect order is `1010101`.


In [ ]:
counts = run_local(afm_chain(5.2))
for bits, n in sorted(counts.items(), key=lambda kv: -kv[1])[:5]:
    print(f'{bits}  {n:>4}  {100 * n / SHOTS:5.1f}%')


## The sweep

Now change only the spacing. Everything else, including the drive and therefore `Rb`,
stays fixed. This takes a couple of minutes on a Colab CPU.


In [ ]:
AFM = '1' + '01' * ((N_ATOMS - 1) // 2)      # 1010101

header = f"{'spacing':>8}{'Rb/a':>7}{'top':>10}{'count':>7}{'AFM %':>8}{'distinct':>10}"
print(header)
print('-' * len(header))

for spacing in (5.0, 5.2, 5.6, 6.0, 6.4, 7.0, 8.0):
    counts = run_local(afm_chain(spacing))
    top_bits, top_n = max(counts.items(), key=lambda kv: kv[1])
    afm_pct = 100 * counts.get(AFM, 0) / SHOTS
    print(f'{spacing:>8.1f}{Rb / spacing:>7.2f}{top_bits:>10}{top_n:>7}'
          f'{afm_pct:>8.1f}{len(counts):>10}')


## What that shows

Three things worth reading off the table.

**The order decays, it does not switch off.** Between `Rb/a` of 1.28 and 1.00, a spacing
change of well under a micrometre, the alternating state falls from about nine runs in ten
to under four.

**The last column is disorder made quantitative.** It counts how many distinct bitstrings
appeared at all. Tight spacing gives around a dozen: one answer plus a few single-atom
defects. As the blockade weakens it climbs past seventy, which is the chain running out of
any reason to prefer one arrangement over another.

**At 8.0 um something else happens.** The dominant outcome is no longer `1010101` but
`1111111`: every atom excited, and the alternating state appears in zero shots. That is
the blockade failing completely rather than weakening. No atom suppresses any other, so
the sweep simply excites all of them.

That last row is the one to remember. A sharply peaked, highly reproducible, completely
wrong answer looks exactly like a correct one if you only check whether the distribution
is sharp.


## Why 5.0 um is the tightest row

It is not a choice. The device refuses atoms closer than 5 um, because tweezers that close
cannot hold them separately.

You also cannot buy more `Rb/a` by driving harder: the blockade radius *shrinks* as the
drive grows. The window is bounded by physics on one side and by hardware on the other,
and at this drive the whole interesting range sits between 5 and 8 um.


## Optional: run it on the service and get a certificate

Everything above ran on this machine. Running the same sequence on ZKSF returns the same
kind of result plus a public certificate stating what was and was not approximated.

Get a token from [app.zksf.org](https://app.zksf.org) using Copy API token. This is the
only part of the notebook that needs an account.


In [ ]:
!pip install -q qsim-sdk


In [ ]:
import qsim_sdk

TOKEN = ''            # paste your token here

if TOKEN:
    client = qsim_sdk.Client(token=TOKEN)
    job = client.run_sequence(afm_chain(5.2), shots=1000)
    counts = job['result']['counts']
    print(sorted(counts.items(), key=lambda kv: -kv[1])[:3])
    print(job['result']['error_info'])
else:
    print('Set TOKEN above to run this cell.')


A certificate from one such run is published at
[cb6bf9ecb491466d](https://api.zksf.org/certify/cb6bf9ecb491466d): 909 shots of `1010101`
out of 1000, on the exact-evolution engine, with the sequence hash and the full
distribution. It also states its own limit. The state is integrated in full with no
truncation, which is why there is no approximation error to report, and also why the
register is capped at 14 atoms. Real neutral-atom hardware runs 100.

Full write-up:
[Rydberg Blockade and Antiferromagnetic Order](https://zksf.org/blog/rydberg-blockade-antiferromagnetic-order/)
